In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())

2.2.2


In [2]:
# =========================
# Pipeline: CDO mergetime per variable -> final CDO merge
# =========================
import os
import glob

# Paths
base_path    = "/work/uc1275/u301827/02_MSE/full_midlatitude/raw"
scratch_path = "/scratch/u/u301827/full_midlatitude/"
out_file     = "/work/uc1275/u301827/02_MSE/full_midlatitude/era5_midlatitudes_JJA_compiled.nc"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(os.path.dirname(out_file), exist_ok=True)

vars_to_merge = [
    "tasmax", "q", "t", "z", "sp", "2d", "blh", "swvl1",
]

def cdo_mergetime(files, out_nc):
    cdo = Cdo()
    if not files:
        return None
    files = sorted(files)
    in_str = " ".join(files)
    if os.path.exists(out_nc):
        os.remove(out_nc)
    cdo.mergetime(input=in_str, output=out_nc, options="-O -f nc")
    return out_nc


def build_merged_filepaths_for_var(var):
    """
    Return paths to intermediate (scratch) merged files for:
      - daily merged file
      - at_tasmax merged file (if exists)
      - dailymax merged file (ONLY for t)
    """
    var_path = os.path.join(base_path, var)
    if not os.path.isdir(var_path):
        print(f"[WARN] Missing folder: {var_path}")
        return ([], None), ([], None), ([], None)

    if var == "tasmax":
        daily_files = glob.glob(os.path.join(var_path, "tasmax_*_midlatitudes.nc"))
        daily_out   = os.path.join(scratch_path, "tasmax_merged.nc")
        return (sorted(daily_files), daily_out), ([], None), ([], None)

    daily_files = glob.glob(os.path.join(var_path, f"{var}_????-??.nc"))
    at_files    = glob.glob(os.path.join(var_path, f"{var}_????-??_at_tasmax.nc"))

    daily_out = os.path.join(scratch_path, f"{var}_merged.nc")
    at_out    = os.path.join(scratch_path, f"{var}_merged_at_tasmax.nc")

    # NEW: only for t
    if var == "t":
        dm_files = glob.glob(os.path.join(var_path, f"{var}_????-??_dailymax.nc"))
        dm_out   = os.path.join(scratch_path, f"{var}_merged_dailymax.nc")
    else:
        dm_files, dm_out = [], None

    return (sorted(daily_files), daily_out), (sorted(at_files), at_out), (sorted(dm_files), dm_out)


# =========================
# 1) CDO: mergetime per variable (daily / at_tasmax / dailymax-for-t)
# =========================
merged_products = []  # list of (path, kind, var)

for var in vars_to_merge:
    print(f"[INFO] CDO mergetime: {var}")

    (daily_files, daily_out), (at_files, at_out), (dm_files, dm_out) = build_merged_filepaths_for_var(var)

    if daily_files:
        outp = cdo_mergetime(daily_files, daily_out)
        if outp:
            merged_products.append((outp, "daily", var))
            print(f"  -> wrote {outp}")
    else:
        print(f"  [WARN] No daily files found for {var}")

    if at_files:
        outp = cdo_mergetime(at_files, at_out)
        if outp:
            merged_products.append((outp, "at_tasmax", var))
            print(f"  -> wrote {outp}")

    # NEW: dailymax only for t
    if var == "t" and dm_files:
        outp = cdo_mergetime(dm_files, dm_out)
        if outp:
            merged_products.append((outp, "dailymax", var))
            print(f"  -> wrote {outp}")
    elif var == "t":
        print(f"  [WARN] No dailymax files found for t")



[INFO] CDO mergetime: tasmax
  -> wrote /scratch/u/u301827/full_midlatitude/tasmax_merged.nc
[INFO] CDO mergetime: q
  -> wrote /scratch/u/u301827/full_midlatitude/q_merged_at_tasmax.nc
[INFO] CDO mergetime: t
  -> wrote /scratch/u/u301827/full_midlatitude/t_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/t_merged_at_tasmax.nc
  -> wrote /scratch/u/u301827/full_midlatitude/t_merged_dailymax.nc
[INFO] CDO mergetime: z
  -> wrote /scratch/u/u301827/full_midlatitude/z_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/z_merged_at_tasmax.nc
[INFO] CDO mergetime: sp
  -> wrote /scratch/u/u301827/full_midlatitude/sp_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/sp_merged_at_tasmax.nc
[INFO] CDO mergetime: 2d
  -> wrote /scratch/u/u301827/full_midlatitude/2d_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitude/2d_merged_at_tasmax.nc
[INFO] CDO mergetime: blh
  -> wrote /scratch/u/u301827/full_midlatitude/blh_merged.nc
  -> wrote /scratch/u/u301827/full_midlatitud

In [3]:
# =========================
# 2) Prepare renamed temp files for final merge
# =========================
cdo = Cdo()
tmp_files_for_merge = []

for path, kind, var in merged_products:
    print(f"[INFO] Preparing for CDO merge: {path}")

    if var == "tasmax":
        # keep tasmax + tasmax_hour as-is (no rename)
        tmp_files_for_merge.append(path)
        continue

    if kind == "daily":
        # keep just var as var
        tmp = os.path.join(scratch_path, f"keep_{var}.nc")
        cdo.selname(var, input=path, output=tmp)
        tmp_files_for_merge.append(tmp)

    elif kind == "at_tasmax":
        # rename var -> var_at_tasmax
        tmp = os.path.join(scratch_path, f"{var}_at_tasmax.nc")
        cdo.chname(
            f"{var},{var}_at_tasmax",
            input=f"-selname,{var} {path}",
            output=tmp
        )
        tmp_files_for_merge.append(tmp)

    elif kind == "dailymax":
        # rename var -> var_dailymax
        tmp = os.path.join(scratch_path, f"{var}_dailymax.nc")
        cdo.chname(
            f"{var},{var}_dailymax",
            input=f"-selname,{var} {path}",
            output=tmp
        )
        tmp_files_for_merge.append(tmp)

        # OPTIONAL: if your dailymax file also contains t_hour, include it too
        # (in my earlier version it's named f"{var}_hour" e.g. "t_hour")
        tmp_hour = os.path.join(scratch_path, f"{var}_dailymax_hour.nc")
        try:
            cdo.chname(
                f"{var}_hour,{var}_dailymax_hour",
                input=f"-selname,{var}_hour {path}",
                output=tmp_hour
            )
            tmp_files_for_merge.append(tmp_hour)
        except Exception:
            # if variable doesn't exist, ignore
            pass


# =========================
# 3) Merge all variables into one dataset and sort by time
# =========================
tmp_merged = os.path.join(scratch_path, "ds_merged_allvars.nc")
cdo.merge(input=" ".join(tmp_files_for_merge), output=tmp_merged)

cdo.sorttimestamp(input=tmp_merged, output=out_file)

print(f"[DONE] Final dataset written to: {out_file}")
print(f"[INFO] Intermediate merged files are in: {scratch_path}")

[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/tasmax_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/q_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/q_merged_at_tasmax.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/t_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/t_merged_at_tasmax.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/t_merged_dailymax.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/z_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/z_merged_at_tasmax.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/sp_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/sp_merged_at_tasmax.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlatitude/2d_merged.nc
[INFO] Preparing for CDO merge: /scratch/u/u301827/full_midlati

In [4]:
import xarray as xr
ds = xr.open_dataset("/work/uc1275/u301827/02_MSE/full_midlatitude/derived_monthly_metpy/derived_194006.nc")

In [2]:
ds

<xarray.Dataset>
Dimensions:          (bnds: 2, lat: 89, lon: 1280, nhyi: 138, nhym: 137, time: 30)
Coordinates:
  * time             (time) datetime64[ns] 1940-06-01 1940-06-02 ... 1940-06-30
  * lon              (lon) float64 0.0 0.2812 0.5625 ... 359.2 359.4 359.7
  * lat              (lat) float64 64.78 64.5 64.22 63.93 ... 40.61 40.33 40.05
    plev             float64 ...
Dimensions without coordinates: bnds, nhyi, nhym
Data variables: (12/29)
    hyai             (nhyi) float64 ...
    hybi             (nhyi) float64 ...
    hyam             (nhym) float64 ...
    hybm             (nhym) float64 ...
    depth_bnds       (bnds) float64 ...
    tasmax           (time, lat, lon) float32 ...
    ...               ...
    mse_sat          (time, lat, lon) float32 ...
    t_bound          (time, lat, lon) float32 ...
    t_bound_mse      (time, lat, lon) float32 ...
    TLCL             (time, lat, lon) float32 ...
    zLCL             (time, lat, lon) float32 ...
    pLCL             (time, lat, lon) float32 ...